<a href="https://colab.research.google.com/github/mentor-pranaya/Cricket-Player-Performance-Prediction/blob/akshaya/week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np


In [3]:
deliveries_df = pd.read_csv('/content/drive/MyDrive/cricket_ml/deliveries.csv')
matches_df = pd.read_csv('/content/drive/MyDrive/cricket_ml/matches.csv')


In [4]:
merged_df = deliveries_df.merge(
    matches_df[['id','season','date','team1','team2','venue']],
    left_on='match_id',
    right_on='id',
    how='left'
)


In [5]:
player_match = merged_df.groupby(
    ['season','match_id','date','batter']
).agg(
    runs=('batsman_runs','sum'),
    balls=('ball','count')
).reset_index()

player_match.head()


,season,match_id,date,batter,runs,balls
0,2007/08,335982,2008-04-18,AA Noffke,9,12
1,2007/08,335982,2008-04-18,B Akhil,0,2
2,2007/08,335982,2008-04-18,BB McCullum,158,77
3,2007/08,335982,2008-04-18,CL White,6,10
4,2007/08,335982,2008-04-18,DJ Hussey,12,12


In [6]:
player_match['strike_rate'] = (player_match['runs'] / player_match['balls']) * 100
player_match.head()


,season,match_id,date,batter,runs,balls,strike_rate
0,2007/08,335982,2008-04-18,AA Noffke,9,12,75.000000
1,2007/08,335982,2008-04-18,B Akhil,0,2,0.000000
2,2007/08,335982,2008-04-18,BB McCullum,158,77,205.194805
3,2007/08,335982,2008-04-18,CL White,6,10,60.000000
4,2007/08,335982,2008-04-18,DJ Hussey,12,12,100.000000


In [7]:
player_match['date'] = pd.to_datetime(player_match['date'])
player_match = player_match.sort_values(['batter','date'])


In [8]:
player_match['avg_runs_last_5'] = (
    player_match
    .groupby('batter')['runs']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [9]:
player_match['avg_sr_last_5'] = (
    player_match
    .groupby('batter')['strike_rate']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [10]:
player_match['career_avg_runs'] = (
    player_match
    .groupby('batter')['runs']
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)


In [11]:
player_match['matches_played'] = (
    player_match
    .groupby('batter')
    .cumcount() + 1
)


In [12]:
features_df = player_match[[
    'season',
    'batter',
    'matches_played',
    'avg_runs_last_5',
    'avg_sr_last_5',
    'career_avg_runs',
    'runs'
]]

features_df.head()


,season,batter,matches_played,avg_runs_last_5,avg_sr_last_5,career_avg_runs,runs
4299,2012,A Ashish Reddy,1,10.00,100.0,10.00,10
4390,2012,A Ashish Reddy,2,6.50,100.0,6.50,3
4496,2012,A Ashish Reddy,3,7.00,100.0,7.00,8
4699,2012,A Ashish Reddy,4,7.75,137.5,7.75,10
4747,2012,A Ashish Reddy,5,7.00,126.0,7.00,4


In [13]:
features_df.fillna(0, inplace=True)


/tmp/ipython-input-2365536963.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_df.fillna(0, inplace=True)


In [14]:
features_df.to_csv(
    '/content/drive/MyDrive/cricket_ml/batsman_features.csv',
    index=False
)
